# Statistical SL/TP Backtest

This notebook is the second part of the requested work:

- use MAE/MFE statistics from the May 25-29 trade sample
- derive defensible SL/TP candidates from the distribution
- run the supplied MA Cross tick replay/backtest code with the original settings and the statistical settings
- compare whether the statistical settings improved the replay outcome

The converted backtest script is saved as `ma_cross_backtest.py`.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path(r"C:\Users\HP\Documents\Codex\2026-05-31\i-did-validation-using-vectorbt-pro")
PYTHON = Path(r"C:\Users\HP\.cache\codex-runtimes\codex-primary-runtime\dependencies\python\python.exe")
BACKTEST = ROOT / "outputs" / "ma_cross_backtest.py"

TICK_FILE = ROOT / "work" / "backtest_week_2026-05-25_29" / "XAUUSD_week.csv"
REPORT_FILE = ROOT / "work" / "backtest_week_2026-05-25_29" / "ReportHistory-week.xlsx"

assert BACKTEST.exists(), BACKTEST
assert TICK_FILE.exists(), TICK_FILE
assert REPORT_FILE.exists(), REPORT_FILE


## Statistical Rule Used

The original settings are:

- SL: 250 points
- TP: 350 points

From the MAE/MFE notebook:

- winning-trade MAE 90th percentile: about 203 points
- winning-trade MAE 95th percentile: about 220 points
- winning-trade MFE 75th percentile: about 382 points

Two non-bruteforce candidates were checked:

- `SL 205 / TP 380`: preserve about 90% of historical winners by MAE and set TP near the 75th percentile of winner MFE.
- `SL 220 / TP 380`: more conservative SL, preserving about 95% of historical winners by MAE.


In [ ]:
def run_backtest(name: str, sl_points: int, tp_points: int) -> dict:
    out_dir = ROOT / "work" / name
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        str(PYTHON),
        str(BACKTEST),
        "--day-dir", str(out_dir),
        "--raw-ticks", str(TICK_FILE),
        "--report", str(REPORT_FILE),
        "--scope", "full-range",
        "--stop-loss-points", str(sl_points),
        "--take-profit-points", str(tp_points),
        "--volume", "0.01",
    ]
    result = subprocess.run(cmd, text=True, capture_output=True, check=False)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return {"name": name, "sl": sl_points, "tp": tp_points, "returncode": result.returncode, "stdout": result.stdout}


# Uncomment to rerun from inside Jupyter.
# original = run_backtest("backtest_week_original_sl250_tp350", 250, 350)
# candidate_90 = run_backtest("backtest_week_stats_sl205_tp380", 205, 380)
# candidate_95 = run_backtest("backtest_week_stats_sl220_tp380", 220, 380)


## Results From This Run

| Setting | SL | TP | Replay trades | Replay PnL | Result vs original replay |
|---|---:|---:|---:|---:|---:|
| Original | 250 | 350 | 362 | 4.17 | baseline |
| Stats candidate, 90% winner-MAE SL | 205 | 380 | 362 | -11.80 | -15.97 |
| Stats candidate, 95% winner-MAE SL | 220 | 380 | 362 | -11.41 | -15.58 |

Conclusion: on this May 25-29 sample, the statistical SL/TP candidates did **not** improve the replay outcome. The tighter statistical SL reduced outcome materially versus the original 250/350 settings.

Important validation note: the supplied replay code produced the same trade count as MT5 for the original settings, but the sequence diverged at trade 110. The original replay PnL was 4.17 versus MT5 report PnL 23.44. That means the backtest engine is usable for this requested comparison, but the result should be presented as a replay comparison, not as a perfectly matched MT5 clone.


In [ ]:
results = [
    {"setting": "Original", "sl": 250, "tp": 350, "replay_trades": 362, "replay_pnl": 4.17},
    {"setting": "Stats q90 winner-MAE", "sl": 205, "tp": 380, "replay_trades": 362, "replay_pnl": -11.80},
    {"setting": "Stats q95 winner-MAE", "sl": 220, "tp": 380, "replay_trades": 362, "replay_pnl": -11.41},
]
results
